# Day 6 — Extra exercises: logistic regression, Simpson's paradox and Bayes' rule

**For two kinds of reader.** If you **missed Day 6** (after informing the lecturer in advance), this notebook is part of your make-up work. If you attended and want **more practice**, work through any part at your own pace; nothing here is required for you.

**What a make-up submission for Day 6 contains** (see `../day-6-makeup.md`):

- **A1–A4 and B1–B3** of this notebook;
- your own solutions to the core in-class exercises in `../day-6-exercises-morning.ipynb` and `../day-6-exercises-afternoon.ipynb`.

**C-problems are optional challenges** for everyone. The make-up submission is graded on the 0–4 in-class scale and counts as your in-class score for the missed day.

**Topics.** Part A: a yes/no outcome; probability, odds and log-odds; priors on the log-odds scale; a logistic regression with a design matrix in Stan; odds ratios, predictions and a grouped check. Part B: Simpson's paradox; marginal versus conditional comparisons; a fair overall comparison; what to adjust for. Part C: Bayes' rule with a careful look at how the data arose.

**Background.** Read the Day 6 slides first: `../day-6-slides-morning.pdf` (logistic regression) and `../day-6-slides-afternoon.pdf` (Simpson's paradox, Monty Hall, the workflow). Each task names the slides it builds on. The demos in `../demos/` show the same steps on the slides' example.

**How to work.** Open in JupyterLab, select **R (bayes)**, run the cells in order from a fresh kernel, from this `extra/` folder. Hints are hidden under **Hint**; try first. Write explanations in your own words in the *Your answer* cells: numbers alone are not enough. All data are **synthetic teaching data**; never report them as real measurements.

**Stan.** Every fit runs live. `models/seedlings.stan` is a **template**: complete the lines marked `TODO` (open it in JupyterLab's text editor) before fitting. If your installation fails, use UCloud and tell the lecturer early.

**Submitting (make-up).** Submit this notebook with outputs, the `.stan` file you edited, and your written answers, together with the in-class exercises, through the announced route by the announced deadline (normally within one week of the missed day). Discussing ideas is fine; the code and explanations must be your own. You may use AI assistants without declaring it, but you should be able to explain every line.

## Setup

Run this cell first.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # dplyr for summaries, ggplot2 for figures
library(cmdstanr)    # Stan from R
set.seed(2026)
options(width = 100, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")
theme_set(theme_minimal(base_size = 14) + theme(legend.position = "top"))
DATA <- "data"; MODELS <- "models"     # run from the extra/ folder
shade_colours <- c("no shade net" = "navy", "shade net" = "tomato")

## Part A — Logistic regression

### The data

A tree nursery planted **160 seedlings**: 20 at each of four watering levels (**1, 2, 3 and 4 litres per week**), each without and with a **shade net**. After one season it recorded whether each seedling **survived** (`data/seedlings-watering.csv`: `seedling`, `water`, `shade`, `survived`).

In [ ]:
seed <- read_csv(file.path(DATA, "seedlings-watering.csv"), show_col_types = FALSE) |>
  mutate(w = water - 2.5,                                  # centred watering (litres per week)
         net = factor(if_else(shade == 1, "shade net", "no shade net"), levels = c("no shade net", "shade net")))
head(seed)

### A1. Survived or not

*Slides "Hit or miss?", "A yes/no outcome" and "Probability, odds and log-odds" (morning).* The model, with wᵢ = water − 2.5 and sᵢ = 1 with a shade net:

survivedᵢ ~ Bernoulli(pᵢ),   logit(pᵢ) = α + β_w · wᵢ + β_s · sᵢ.

1. Make a table of survivors out of seedlings for each watering level and net, and plot the share surviving against water, one colour per net.
2. At 2 litres without a net, compute the share surviving, its odds and its log-odds.
3. In one sentence each: what do α, β_w and β_s mean? Why is it convenient to centre water at 2.5?

In [ ]:
# seed |> group_by(water, net) |> summarise(survivors = sum(survived), seedlings = n(), share = mean(survived), .groups = "drop")

<details><summary><b>Hint</b> (click to open)</summary>

`qlogis(p)` is the log-odds log(p / (1 − p)); `plogis()` goes back.

</details>

**Your answer:** *(write here)*

### A2. Priors on the log-odds scale

*Slides "Priors on the log-odds scale", "Priors for this model" and "Prior predictive: curves of P(hit)" (morning).* Proposed priors: α ~ Normal(0, 1.5), β_w ~ Normal(0, 1) per litre, β_s ~ Normal(0, 1).

1. What odds ratios per litre and for the net do these priors allow (±2 prior SDs)? Are they reasonable for watering and shading young trees?
2. Draw 4000 values of α from Normal(0, 1.5) and from Normal(0, 10), and give the share of implied P(survive) below 0.01 or above 0.99 for each.
3. Plot 50 prior curves of P(survive) against water (0.5 to 4.5 litres), with and without a net.

In [ ]:
S <- 4000
# exp(c(-2, 2))

<details><summary><b>Hint</b> (click to open)</summary>

P(survive) is below 0.01 or above 0.99 exactly when |log-odds| > `qlogis(0.99)` = 4.6, so with only α you can count `abs(alpha) > qlogis(0.99)`.

</details>

**Your answer:** *(write here)*

### A3. The logistic regression in Stan

*Slides "The design matrix", "The logistic model in Stan" and "The fit" (morning).*

1. Complete the template `models/seedlings.stan` (the prior of β, the likelihood, `p_new` and `survived_rep`).
2. Build X with the columns w and shade, and X_new with these four cases: 1 litre without a net, 1 litre with a net, 4 litres without a net, 4 litres with a net. Fit the model (4 chains, `seed = 2026`, `prior_sd = c(1, 1)`) and check the diagnostics.
3. Report the odds ratio per extra litre and for the net, with 90% intervals, and P(β_s > 0 | y).

In [ ]:
# X <- cbind(w = seed$w, shade = seed$shade)
# cases <- tibble(water = c(1, 1, 4, 4), shade = c(0, 1, 0, 1))
# X_new <- cbind(w = cases$water - 2.5, shade = cases$shade)
# stan_data <- list(N = nrow(seed), K = 2, X = X, survived = seed$survived, prior_sd = c(1, 1), M = nrow(X_new), X_new = X_new)
# mod <- cmdstan_model(file.path(MODELS, "seedlings.stan"))

<details><summary><b>Hint</b> (click to open)</summary>

The template is the course's `logistic.stan` with the outcome called `survived`: `beta ~ normal(0, prior_sd);`, `survived ~ bernoulli_logit(alpha + X * beta);`, `p_new = inv_logit(alpha + X_new * beta);`, `survived_rep = bernoulli_logit_rng(alpha + X * beta);`.

</details>

**Your answer:** *(write here)*

### A4. Predictions and a grouped check

*Slides "Predicted probabilities for chosen cases" and "A predictive check for yes/no data" (morning).*

1. Report P(survive) with 90% intervals for the four cases.
2. The nursery plants **20 more seedlings at 2 litres with a net**. Give the 90% interval of the number that survive.
3. Grouped check: for each of the 8 groups (water × net), compare the observed number of survivors with the 90% range of replicated numbers. Any misfit?

In [ ]:
# cases |> mutate(mean = fit$summary("p_new")$mean, ...)
# p_2_net <- plogis(draws$alpha + draws$`beta[1]` * (2 - 2.5) + draws$`beta[2]`)

<details><summary><b>Hint</b> (click to open)</summary>

For task 2, P(survive) at 2 litres with a net is `plogis(alpha + beta_w * (2 - 2.5) + beta_s)` for each draw; then `rbinom(S, 20, p)` gives one possible number of survivors per draw. Task 3 is the Day 6 morning check (Exercise A4) with water instead of target.

</details>

**Your answer:** *(write here)*

## Part B — Simpson's paradox

### The data

The same nursery company has **two sites**: a cold, exposed site **north** and a mild site **south**, with 200 seedlings each (`data/seedlings-two-sites.csv`: `seedling`, `site`, `shade`, `survived`). The staff in the north put shade nets on most of their seedlings (150 of 200), those in the south on few (50 of 200).

In [ ]:
sites <- read_csv(file.path(DATA, "seedlings-two-sites.csv"), show_col_types = FALSE) |>
  mutate(north = as.integer(site == "north"),
         net = factor(if_else(shade == 1, "shade net", "no shade net"), levels = c("no shade net", "shade net")))
sites |> count(site, net)

### B1. Within each site versus overall

*Slides "Does feedback help? A pilot study", "Within every target versus overall" and "Why it happens" (afternoon).*

1. Make a table of the share surviving for each site and net, and for each net over both sites. Plot both, with a legend.
2. Does the net help within each site? Overall? Explain the difference.
3. With the mix of sites of the seedlings **without** a net, what would the overall share **with** a net be?

In [ ]:
# by_site <- sites |> group_by(group = site, net) |> summarise(survivors = sum(survived), seedlings = n(), .groups = "drop")

<details><summary><b>Hint</b> (click to open)</summary>

Each overall share is a weighted average of the two site shares; the weights are how the seedlings of that group are split between the sites. `count(site)` within each net gives them.

</details>

**Your answer:** *(write here)*

### B2. Two logistic models and a fair comparison

*Slides "Two logistic models, two answers" and "A fair overall comparison" (afternoon).* Use your completed `models/seedlings.stan` with two design matrices:

- Model 1: one column, shade (`prior_sd = 1`);
- Model 2: two columns, north (1 = north site) and shade (`prior_sd = c(1.5, 1)`), with X_new holding the four combinations of site and net.

1. Fit both. Report the 90% interval of the shade coefficient and its odds ratio in each, and P(β_s > 0 | y).
2. From Model 2, average P(survive) over an **equal** mix of the two sites, with and without a net. Give the difference with a 90% interval.

In [ ]:
# data1 <- list(N = nrow(sites), K = 1, X = cbind(shade = sites$shade), survived = sites$survived, prior_sd = 1, M = 0, X_new = matrix(0, 0, 1))

<details><summary><b>Hint</b> (click to open)</summary>

The rows of X_new decide which `p_new[k]` is which: write them down. An equal mix of the two sites is the average of the two sites' predictions.

</details>

**Your answer:** *(write here)*

### B3. What to adjust for

*Slide "Which comparison answers the question?" (afternoon).*

1. Draw (on paper, or in words) the diagram of site, net and survival. Why must the comparison hold the site fixed?
2. At the end of the season the staff also recorded **leaf damage** on each seedling, which the net reduces and which itself lowers survival. Should leaf damage be a column of X when you estimate the effect of the net on survival? Why or why not?
3. Name one thing about how these data arose that no model can fix, and what the nursery could do next year to get a cleaner answer.

**Your answer:** *(write here)*

**Reflection (optional).** In 80–120 words: describe the steps of the modelling workflow you used in Parts A and B (*slide "The modelling workflow"*).

**Your reflection:** *(write here)*

## Part C — Optional challenges

### C1. The three prisoners

*Slides "The Monty Hall problem" to "Change the host's rule" (afternoon).* Three prisoners, A, B and C, know that one of them, chosen at random, will be pardoned. A asks the guard, who knows, to name one of the **other two** who will **not** be pardoned (if both B and C will not be, the guard picks one of them at random). The guard says "B". Before, A's chance was 1/3. Now?

1. Find P(A pardoned | guard says B) and P(C pardoned | guard says B) by Bayes' rule, in a table like the slide's.
2. Check by simulation (10,000 rounds).
3. How is this the Monty Hall problem in disguise?

In [ ]:
# C1: your code

**Your answer:** *(write here)*

### C2. Does the net matter more for dry seedlings?

Add a third column, w × shade, to the design matrix of Part A (prior SD 0.5 for it) and fit again. Report the coefficient of the new column with a 90% interval, and the odds ratio for the net at 1 litre and at 4 litres. Is there evidence that the effect of the net depends on watering?

In [ ]:
# C2: your code

**Your answer:** *(write here)*